# Phase 06B.03E — Full temporal VQA evaluation

Runs smoke first; full requires locked protocol, feature banks, checkpoints and exact registry.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import fixed_tile_allocation,sha256_file
from phase06b_common import EXPERIMENT_ROLES,validate_full_temporal_predictions,validate_temporal_experiment_registry
RUN_SCOPE="smoke"; OUT=ROOT/"outputs/phase06b/temporal_vqa"/RUN_SCOPE; OUT.mkdir(parents=True,exist_ok=True)
PROTOCOL=ROOT/"outputs/phase06b/novelty_protocol/novelty_protocol.json"; REGISTRY=ROOT/"outputs/phase06b/experiments/novelty_experiment_registry.json"; IDS=ROOT/"data/splits/phase01/validation_sample_ids.json"


In [ ]:
if not PROTOCOL.is_file() or not REGISTRY.is_file():
    save_json(OUT/"PHASE06B_03E_STATUS.json",{"status":"awaiting_locked_protocol_registry","scope":RUN_SCOPE,"scientific_complete":False}); raise RuntimeError("Protocol and registry must be locked before inference")
protocol=json.loads(PROTOCOL.read_text()); registry=validate_temporal_experiment_registry(json.loads(REGISTRY.read_text()),protocol=protocol)
ids=json.loads(IDS.read_text()); arms={arm["name"]:arm for arm in registry["arms"]}
# Inference outputs are arm-isolated and must already have raw model responses plus all candidate/selection provenance.
for name,arm in arms.items():
    path=OUT/name/"predictions.csv"
    if not path.is_file(): raise RuntimeError(f"Missing {RUN_SCOPE} prediction artifact for {name}")
    if RUN_SCOPE=="full": validate_full_temporal_predictions(pd.read_csv(path),ids,arm=arm)
status="complete" if RUN_SCOPE=="full" and len(ids)==298 else "smoke_complete"
save_json(OUT/"PHASE06B_03E_STATUS.json",{"status":status,"scope":RUN_SCOPE,"rows_per_arm":298 if RUN_SCOPE=="full" else None,"scientific_complete":RUN_SCOPE=="full"})
